# 05 — Full Pipeline

End-to-end run: set a bounding box and get speed limit estimates for every
Overture segment in the area, evaluated against Overture's own normalized
speed limit values.

**Usage:**
1. Set `BBOX` and `TOKEN`
2. Run all cells
3. Find the final GeoParquet in `speed_limit_estimates.parquet`

In [1]:
import os
import geopandas as gpd
from dotenv import load_dotenv
from slc import consensus, evaluate, fetch, match, snap, split, viz

load_dotenv()
TOKEN = os.environ['MAPILLARY_ACCESS_TOKEN']

# ── Configuration ──────────────────────────────────────────────────────────
# North Boulder, CO ~800m area
BBOX = (-105.262, 40.011, -105.252, 40.018)

SNAP_MAX_DIST_M      = 10.0   # reject signs >10m from sequence
SNAP_MAX_HDG_DIFF    = 30.0
SPLIT_BEARING_THRESH = 60.0
SPLIT_WINDOW_M       = 80.0
MATCH_MAX_DIST_M     = 25.0

# Sign speed unit — "mph" for US/UK, "kmh" for most other countries.
# Mapillary sign values are in the native unit of the country; when set
# to "kmh" the pipeline converts to mph automatically.
SPEED_UNIT = "mph"

# Plausible speed ranges (mph) per Overture road class for Colorado.
# Used to gate "net new" candidates — estimates outside these ranges
# are flagged as implausible.
PLAUSIBLE_SPEED_RANGES = {
    "motorway":     (55, 75),
    "trunk":        (40, 65),
    "primary":      (25, 55),
    "secondary":    (25, 45),
    "tertiary":     (15, 40),
    "residential":  (15, 30),
    "unclassified": (15, 45),
    "service":      (5, 25),
    "living_street": (5, 20),
    "footway":      (None, None),  # pedestrian — skip
    "cycleway":     (None, None),
    "path":         (None, None),
    "pedestrian":   (None, None),
    "steps":        (None, None),
}
# ───────────────────────────────────────────────────────────────────────────

In [2]:
print('1/5  Fetching Mapillary data...')
signs     = fetch.fetch_mapillary_signs(BBOX, TOKEN, unit=SPEED_UNIT)
images    = fetch.fetch_mapillary_images(BBOX, TOKEN)
sequences = fetch.build_sequences(images)
print(f'     Signs: {len(signs)}  |  Sequences: {len(sequences)}')

if not signs.empty:
    print(f'\n     Sign details:')
    for _, s in signs.iterrows():
        stype = s.get("sign_type", "?")
        print(f'       • {s["speed_mph"]} mph  [{stype}]  ({s["raw_value"]})  '
              f'at ({s.geometry.x:.5f}, {s.geometry.y:.5f})')
    # Sign type breakdown
    print(f'\n     Sign type breakdown:')
    for stype, count in signs['sign_type'].value_counts().items():
        print(f'       {stype}: {count}')

print('\n2/5  Fetching Overture segments (includes normalized speed limits)...')
overture_raw = fetch.fetch_overture_segments(BBOX)
overture     = fetch.extract_overture_speed_limits(overture_raw)
has_sl = overture["speed_limit_value"].notna().sum()
no_sl  = overture["speed_limit_value"].isna().sum()
print(f'     Segments: {len(overture)}  |  With speed limit: {has_sl}  |  Without: {no_sl}')

# Road class breakdown
if 'class' in overture.columns:
    print(f'\n     Road class breakdown:')
    for cls, grp in overture.groupby('class'):
        with_sl = grp['speed_limit_value'].notna().sum()
        without_sl = grp['speed_limit_value'].isna().sum()
        print(f'       {cls:20s}  total={len(grp):4d}  with_SL={with_sl:3d}  without={without_sl:3d}')

1/5  Fetching Mapillary data...
     Signs: 16  |  Sequences: 142

     Sign details:
       • 35 mph  [standard]  (regulatory--maximum-speed-limit-35--g2)  at (-105.26185, 40.01730)
       • 35 mph  [standard]  (regulatory--maximum-speed-limit-35--g2)  at (-105.25857, 40.01389)
       • 35 mph  [standard]  (regulatory--maximum-speed-limit-35--g2)  at (-105.26052, 40.01722)
       • 35 mph  [standard]  (regulatory--maximum-speed-limit-35--g2)  at (-105.26118, 40.01754)
       • 35 mph  [standard]  (regulatory--maximum-speed-limit-35--g2)  at (-105.26128, 40.01755)
       • 35 mph  [standard]  (regulatory--maximum-speed-limit-35--g2)  at (-105.26145, 40.01754)
       • 35 mph  [standard]  (regulatory--maximum-speed-limit-35--g2)  at (-105.25210, 40.01450)
       • 35 mph  [standard]  (regulatory--maximum-speed-limit-35--g2)  at (-105.25875, 40.01405)
       • 35 mph  [standard]  (regulatory--maximum-speed-limit-35--g2)  at (-105.26039, 40.01730)
       • 30 mph  [standard]  (regulatory-

In [3]:
print('3/5  Snapping signs and splitting sequences...')
snapped     = snap.snap_signs_to_sequences(signs, sequences,
                  max_distance_m=SNAP_MAX_DIST_M,
                  max_heading_diff=SNAP_MAX_HDG_DIFF)
print(f'     Snapped sign→sequence pairs: {len(snapped)}')
if not snapped.empty:
    for _, r in snapped.iterrows():
        print(f'       • sign {r["sign_id"]} → seq {r["sequence_id"][:12]}…  '
              f'snap={r["snap_distance_m"]:.1f}m  along={r["distance_along_m"]:.0f}m  '
              f'{r["speed_mph"]}mph')

split_edges = split.split_all_sequences(sequences, snapped,
                  bearing_threshold_deg=SPLIT_BEARING_THRESH,
                  window_m=SPLIT_WINDOW_M)
labeled = split_edges[split_edges['speed_mph'].notna()]
unlabeled = split_edges[split_edges['speed_mph'].isna()]
print(f'\n     Split edges: {len(split_edges)}  (labeled: {len(labeled)}, unlabeled: {len(unlabeled)})')

# ASCII diagram of splits per sequence
if not split_edges.empty:
    print(f'\n     Edge map per sequence (— = unlabeled, ## = labeled with speed):')
    for seq_id in split_edges['sequence_id'].unique()[:8]:
        seq_edges = split_edges[split_edges['sequence_id'] == seq_id].sort_values('edge_id')
        bar = ''
        for _, e in seq_edges.iterrows():
            seg_len = max(1, int(e['length_m'] / 50))  # ~1 char per 50m
            if e['speed_mph'] is not None and not (isinstance(e['speed_mph'], float) and e['speed_mph'] != e['speed_mph']):
                bar += f'[{int(e["speed_mph"])}]' + '█' * seg_len
            else:
                bar += '─' * seg_len
            reason = e['split_reason']
            if reason == 'sign':
                bar += '🔵'
            elif reason == 'turn':
                bar += '↩'
        total_m = seq_edges['length_m'].sum()
        print(f'       {seq_id[:12]}… ({total_m:.0f}m): {bar}')

3/5  Snapping signs and splitting sequences...
     Snapped sign→sequence pairs: 89
       • sign 1044499356077727 → seq o2zl9z7hf2df…  snap=7.3m  along=10m  35mph
       • sign 1044499356077727 → seq gcetcf39xbk9…  snap=1.9m  along=11m  35mph
       • sign 1044499356077727 → seq 5q0CC42TQRmv…  snap=6.7m  along=1m  35mph
       • sign 1044499356077727 → seq pkyGFA0NVldr…  snap=8.7m  along=12m  35mph
       • sign 1044499356077727 → seq XliApNR4QDH7…  snap=7.9m  along=8m  35mph
       • sign 368989775720632 → seq 1SmCv7xAUGPb…  snap=7.7m  along=213m  35mph
       • sign 368989775720632 → seq RNdKuSnelz6v…  snap=1.7m  along=147m  35mph
       • sign 368989775720632 → seq hmd9r3yd5bl3…  snap=0.9m  along=643m  35mph
       • sign 368989775720632 → seq o2zl9z7hf2df…  snap=5.5m  along=671m  35mph
       • sign 368989775720632 → seq -s5Rdoa6THir…  snap=5.2m  along=114m  35mph
       • sign 368989775720632 → seq i1ypsrae9s5r…  snap=4.8m  along=92m  35mph
       • sign 368989775720632 → seq m3D

In [4]:
print('4/5  Matching to Overture segments...')
matches   = match.match_edges_to_overture(split_edges, overture,
                max_distance_m=MATCH_MAX_DIST_M)
print(f'     Matches: {len(matches)}')
if not matches.empty:
    print(f'\n     Match details:')
    for _, m in matches.head(10).iterrows():
        print(f'       edge {m["edge_id"]} → overture {str(m["overture_id"])[:16]}…  '
              f'LR=[{m["lr_start"]:.2f}–{m["lr_end"]:.2f}]  '
              f'score={m["score"]:.2f}  {int(m["speed_mph"])}mph')
else:
    # Debug why no matches
    print('\n     ⚠ No matches found. Diagnostics:')
    labeled_edges = split_edges[split_edges['speed_mph'].notna()]
    print(f'       Labeled edges to match: {len(labeled_edges)}')
    if not labeled_edges.empty:
        print(f'       Sample labeled edge bbox: {labeled_edges.total_bounds}')
        print(f'       Overture segments bbox:   {overture.total_bounds}')

estimates = consensus.compute_consensus(matches)
print(f'\n     Consensus estimates: {len(estimates)}')
if not estimates.empty:
    for _, e in estimates.iterrows():
        conflict = ' ⚠CONFLICT' if e.get('has_conflict') else ''
        print(f'       {str(e["overture_id"])[:16]}…  {int(e["speed_mph"])}mph  '
              f'conf={e["confidence"]:.2f}  obs={e["observation_count"]}{conflict}')

4/5  Matching to Overture segments...
     Matches: 356

     Match details:
       edge -s5Rdoa6THirIfNplZmg3A_1 → overture a315258b-0644-46…  LR=[0.16–0.96]  score=0.45  35mph
       edge 1Phnvy7BwMTecrQSAR8DuU_1 → overture 2b1354c3-376c-4a…  LR=[0.43–0.81]  score=0.07  35mph
       edge 1Phnvy7BwMTecrQSAR8DuU_1 → overture 841db016-92c7-49…  LR=[0.00–0.62]  score=0.29  35mph
       edge 1Phnvy7BwMTecrQSAR8DuU_1 → overture 503c0dac-d6ee-46…  LR=[0.45–1.00]  score=0.79  35mph
       edge 1Phnvy7BwMTecrQSAR8DuU_1 → overture f0b89a46-509a-4d…  LR=[0.00–0.58]  score=0.13  35mph
       edge 1Phnvy7BwMTecrQSAR8DuU_1 → overture 6ade0584-2ae2-40…  LR=[0.59–0.85]  score=0.79  35mph
       edge 1SmCv7xAUGPb5fcBNi9HWZ_1 → overture 2ff80f0b-e386-48…  LR=[0.00–0.20]  score=0.12  35mph
       edge 1SmCv7xAUGPb5fcBNi9HWZ_1 → overture a20836af-c8ec-49…  LR=[0.00–0.64]  score=0.20  35mph
       edge 1SmCv7xAUGPb5fcBNi9HWZ_1 → overture 43a53d08-dda1-4a…  LR=[0.18–1.00]  score=0.65  35mph
       edge 1S

In [5]:
print('5/5  Evaluating against Overture speed limits...')
comparison = evaluate.compare_to_overture(estimates, overture)
metrics    = evaluate.compute_metrics(comparison)

if comparison.empty and has_sl == 0:
    print('     ⚠ No Overture ground truth speed limits in this area — skipping accuracy eval')
    print('     (Our estimates are still valid, just can\'t compare to Overture)')
    report = None
else:
    report = evaluate.generate_report(comparison, metrics)
    print(report)

5/5  Evaluating against Overture speed limits...
# Speed Limit Conflation — Evaluation Report

## Coverage
- Total Overture segments with estimates: **14**
- Segments with Overture ground truth: **14**

## Accuracy vs Overture speed limits
- Exact match: **71.4%**
- Within 5 mph: **85.7%**
- Within 10 mph: **92.9%**

## Confusion Matrix (our estimate → Overture value → count)

| Our \ Overture | 20 | 25 | 35 |
| --- | --- | --- | --- |
| 30 | 0 | 0 | 2 |
| 35 | 1 | 1 | 10 |



In [6]:
# Export final dataset
estimates.to_parquet('speed_limit_estimates.parquet', index=False)
print('Saved speed_limit_estimates.parquet')

print(f"\nSummary:")
print(f"  Segments with estimates : {len(estimates)}")
if metrics.get('exact_match_rate') is not None:
    print(f"  Exact match vs Overture : {metrics['exact_match_rate']}")
    print(f"  Within 5 mph vs Overture: {metrics['within_5mph_rate']}")
else:
    print(f"  No Overture ground truth available for accuracy comparison")

# ── Net New Candidates ─────────────────────────────────────────────────────
# Segments where we have an estimate but Overture has NO speed limit
print('\n' + '='*70)
print('NET NEW SPEED LIMIT CANDIDATES')
print('='*70)

if not estimates.empty:
    id_col = 'id' if 'id' in overture.columns else overture.columns[0]

    # Merge estimates with overture to get road class and existing speed limit
    net_new = estimates.merge(
        overture[[id_col, 'speed_limit_value', 'class']].rename(
            columns={id_col: 'overture_id', 'class': 'road_class'}
        ),
        on='overture_id',
        how='left',
    )

    # Net new = estimate exists but Overture has no speed limit
    net_new_candidates = net_new[net_new['speed_limit_value'].isna()].copy()
    confirmed = net_new[net_new['speed_limit_value'].notna()]

    print(f'\n  Estimates matching Overture ground truth: {len(confirmed)}')
    print(f'  Net new candidates (no Overture SL):     {len(net_new_candidates)}')

    if not net_new_candidates.empty:
        # Plausibility gating
        def check_plausible(row):
            cls = row.get('road_class', '')
            speed = row['speed_mph']
            rng = PLAUSIBLE_SPEED_RANGES.get(cls)
            if rng is None:
                return 'unknown_class'
            lo, hi = rng
            if lo is None:
                return 'skip_non_road'
            if lo <= speed <= hi:
                return 'plausible'
            return 'implausible'

        net_new_candidates['plausibility'] = net_new_candidates.apply(check_plausible, axis=1)

        plausible = net_new_candidates[net_new_candidates['plausibility'] == 'plausible']
        implausible = net_new_candidates[net_new_candidates['plausibility'] == 'implausible']
        skipped = net_new_candidates[net_new_candidates['plausibility'].isin(['skip_non_road', 'unknown_class'])]

        print(f'\n  Plausibility gating results:')
        print(f'    ✓ Plausible:   {len(plausible)}')
        print(f'    ✗ Implausible: {len(implausible)}')
        print(f'    ○ Skipped:     {len(skipped)}  (non-road or unknown class)')

        if not plausible.empty:
            print(f'\n  ✓ PLAUSIBLE net new candidates:')
            for _, r in plausible.iterrows():
                rng = PLAUSIBLE_SPEED_RANGES.get(r.get('road_class', ''), ('?', '?'))
                print(f'    {str(r["overture_id"])[:20]:22s}  {int(r["speed_mph"]):3d} mph  '
                      f'class={r.get("road_class", "?"):15s}  '
                      f'range=[{rng[0]}-{rng[1]}]  '
                      f'conf={r["confidence"]:.2f}  obs={r["observation_count"]}')

        if not implausible.empty:
            print(f'\n  ✗ IMPLAUSIBLE candidates (outside expected range):')
            for _, r in implausible.iterrows():
                rng = PLAUSIBLE_SPEED_RANGES.get(r.get('road_class', ''), ('?', '?'))
                print(f'    {str(r["overture_id"])[:20]:22s}  {int(r["speed_mph"]):3d} mph  '
                      f'class={r.get("road_class", "?"):15s}  '
                      f'range=[{rng[0]}-{rng[1]}]  ⚠')

        # Summary by road class
        print(f'\n  Net new by road class:')
        for cls in sorted(net_new_candidates['road_class'].dropna().unique()):
            cls_rows = net_new_candidates[net_new_candidates['road_class'] == cls]
            n_plaus = (cls_rows['plausibility'] == 'plausible').sum()
            n_implaus = (cls_rows['plausibility'] == 'implausible').sum()
            speeds = sorted(cls_rows['speed_mph'].unique())
            print(f'    {cls:20s}  total={len(cls_rows)}  '
                  f'plausible={n_plaus}  implausible={n_implaus}  '
                  f'speeds={[int(s) for s in speeds]}')

        # Export net new plausible candidates
        if not plausible.empty:
            plausible.to_parquet('net_new_candidates.parquet', index=False)
            print(f'\n  Saved {len(plausible)} plausible net new candidates to net_new_candidates.parquet')
    else:
        print('\n  No net new candidates — all estimates have Overture ground truth')
else:
    print('\n  No estimates produced')

Saved speed_limit_estimates.parquet

Summary:
  Segments with estimates : 35
  Exact match vs Overture : 0.7143
  Within 5 mph vs Overture: 0.8571

NET NEW SPEED LIMIT CANDIDATES

  Estimates matching Overture ground truth: 14
  Net new candidates (no Overture SL):     21

  Plausibility gating results:
    ✓ Plausible:   3
    ✗ Implausible: 5
    ○ Skipped:     13  (non-road or unknown class)

  ✓ PLAUSIBLE net new candidates:
    405e1fb4-e749-480a-a     35 mph  class=primary          range=[25-55]  conf=0.28  obs=1
    5c64972a-6c94-4011-9     35 mph  class=tertiary         range=[15-40]  conf=0.42  obs=3
    ae94e554-ef5a-4280-b     35 mph  class=unclassified     range=[15-45]  conf=0.28  obs=1

  ✗ IMPLAUSIBLE candidates (outside expected range):
    3a4a460c-4f0f-4344-8     30 mph  class=service          range=[5-25]  ⚠
    9d1d6296-85b6-495b-9     35 mph  class=service          range=[5-25]  ⚠
    af9467ee-de9b-4f7a-a     35 mph  class=service          range=[5-25]  ⚠
    daa8c

In [7]:
import folium
from folium import PolyLine, Popup, CircleMarker

# Build a map showing: Overture segments (gray), estimates (colored),
# and net new candidates highlighted with dashed lines.
center_lat = (BBOX[1] + BBOX[3]) / 2
center_lon = (BBOX[0] + BBOX[2]) / 2
m = folium.Map(location=[center_lat, center_lon], zoom_start=15)

# Estimate lookup
speed_lookup = {}
if not estimates.empty:
    for _, row in estimates.iterrows():
        speed_lookup[row['overture_id']] = row

# Net new lookup (from cell 6)
net_new_ids = set()
plausible_ids = set()
implausible_ids = set()
if not estimates.empty:
    id_col = 'id' if 'id' in overture.columns else overture.columns[0]
    _nn = estimates.merge(
        overture[[id_col, 'speed_limit_value', 'class']].rename(
            columns={id_col: 'overture_id', 'class': 'road_class'}),
        on='overture_id', how='left')
    for _, r in _nn[_nn['speed_limit_value'].isna()].iterrows():
        net_new_ids.add(r['overture_id'])
        cls = r.get('road_class', '')
        rng = PLAUSIBLE_SPEED_RANGES.get(cls)
        if rng and rng[0] is not None:
            if rng[0] <= r['speed_mph'] <= rng[1]:
                plausible_ids.add(r['overture_id'])
            else:
                implausible_ids.add(r['overture_id'])

# Draw all Overture segments
id_col = 'id' if 'id' in overture.columns else overture.columns[0]
for _, row in overture.iterrows():
    seg_id = row[id_col]
    coords = [(y, x) for x, y in row.geometry.coords]
    est = speed_lookup.get(seg_id)

    if seg_id in plausible_ids:
        # Net new plausible — bright green dashed
        mph = int(est['speed_mph']) if est is not None else '?'
        PolyLine(coords, color='#2ca25f', weight=6, opacity=0.9,
                 dash_array='10 5',
                 popup=Popup(f'✓ NET NEW: {mph} mph (plausible)')).add_to(m)
    elif seg_id in implausible_ids:
        # Net new implausible — orange dashed
        mph = int(est['speed_mph']) if est is not None else '?'
        PolyLine(coords, color='#fd8d3c', weight=6, opacity=0.9,
                 dash_array='10 5',
                 popup=Popup(f'⚠ NET NEW: {mph} mph (implausible)')).add_to(m)
    elif seg_id in net_new_ids:
        # Net new skipped (non-road) — thin gray dashed
        PolyLine(coords, color='#999999', weight=2, opacity=0.5,
                 dash_array='5 5').add_to(m)
    elif est is not None:
        # Has estimate AND Overture ground truth — colored by speed
        mph = int(est['speed_mph'])
        color = viz.speed_color(mph)
        PolyLine(coords, color=color, weight=5, opacity=0.85,
                 popup=Popup(f'{mph} mph (confirmed)')).add_to(m)
    else:
        # No estimate — light gray
        PolyLine(coords, color='#dddddd', weight=1.5, opacity=0.4).add_to(m)

# Signs as markers
if not signs.empty:
    for _, s in signs.iterrows():
        CircleMarker(
            location=[s.geometry.y, s.geometry.x],
            radius=5, color='black', fill=True, fill_color='yellow',
            popup=Popup(f'{s["speed_mph"]} mph ({s["sign_type"]})')
        ).add_to(m)

# Legend
legend_html = '''
<div style="position:fixed; bottom:30px; left:30px; z-index:1000;
     background:white; padding:10px; border:2px solid #ccc; border-radius:5px;
     font-size:12px; line-height:1.6;">
<b>Legend</b><br>
<span style="color:#2ca25f">━━━</span> Net new (plausible)<br>
<span style="color:#fd8d3c">━━━</span> Net new (implausible)<br>
<span style="color:#fee090">━━━</span> Confirmed estimate<br>
<span style="color:#dddddd">━━━</span> No estimate<br>
<span style="color:black">●</span> Speed sign
</div>
'''
m.get_root().html.add_child(folium.Element(legend_html))

m